In [1]:
from dotenv import load_dotenv
load_dotenv()
from groq import Groq

client = Groq()   # GROQ_API_KEY 자동으로 집어감
MODEL = "openai/gpt-oss-120b"

In [3]:
def calculate(expression):
    # 문자열 수식을 받아서 계산 결과를 반환
    # 예: "3 * 24" → 72
    return eval(expression)

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "수식을 입력 받았을 때 수식을 계산하여 반환한다",   # ← LLM이 "언제 이걸 써야 하는지" 판단하는 근거
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "입력 받은 수식, 수식의 예는 다음과 같다. 2*4",   # ← 이 인자에 뭘 넣어야 하는지
                    }
                },
                "required": ["expression"],
            },
        },
    }
]

In [5]:
messages = [
    {"role": "user", "content": "555*10000"}   # ← LLM이 계산을 시도할 만한 질문
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,          # ← 아까 만든 메뉴판을 같이 넘김 (이게 핵심)
)

# LLM이 뭐라고 응답했는지 까보기
msg = response.choices[0].message
print(msg)

ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='User gave "555*10000". Likely they want calculation. Use function calculate.', tool_calls=[ChatCompletionMessageToolCall(id='fc_d30a186c-0482-4ec2-9e46-b9ba749af24e', function=Function(arguments='{"expression":"555*10000"}', name='calculate'), type='function')])


In [6]:
import json

tool_call = msg.tool_calls[0]              # 첫 번째 도구 요청
func_name = tool_call.function.name        # "calculate"
args = json.loads(tool_call.function.arguments)                                  # ← arguments(JSON 문자열)를 딕셔너리로 파싱

# 실제 실행
if func_name == "calculate":
    result = calculate(args["expression"])                   # ← calculate를 args의 expression으로 호출

print("실행 결과:", result)

실행 결과: 5550000


In [7]:
# (1) LLM이 했던 "도구 부르겠다"는 응답을, 대화 기록에 그대로 추가
messages.append(msg)

# (2) 도구 실행 결과를, "tool" 역할로 추가
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,        # 어느 요청에 대한 답인지 연결 (LLM 응답의 그 id)
    "content": str(result),               # 실행 결과 (문자열로)
})

# 이제 결과까지 담긴 messages로 LLM 다시 호출
second_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)
print(second_response.choices[0].message.content)

The result is **5,550,000**.
